# Ch.3 — Calculus: Slopes, Areas, and the Meeting in the Middle

**Running theme.** For our free throw $h(t) = v_{0y}\,t - \tfrac{1}{2}g t^2$, we want two things:
1. The **instantaneous velocity** at any $t$ — the derivative $h'(t)$.
2. The **total rise** from release to apex — the integral $\int_0^T v(t)\,dt$.

**Learning outcomes.**
1. Slide $\Delta t$ toward 0 and watch a secant become a tangent, live.
2. Drag the number-of-rectangles slider and see a Riemann sum converge.
3. Confirm the Fundamental Theorem of Calculus numerically on a real example.
4. Discover the sweet-spot $\Delta t$ for numerical differentiation, and see round-off error kick in.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatLogSlider, IntSlider, FloatText, HBox, VBox, Output, jslink

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (8.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

# Free-throw parameters — same as Ch.1/Ch.2
v0y, g = 7.2, 9.81
h_fn = lambda t: v0y * t - 0.5 * g * t ** 2
v_fn = lambda t: v0y - g * t                  # true analytic derivative
t_apex = v0y / g                              # ≈ 0.734 s

## 2 · Secant becomes tangent

Anchor at $t_0 = 0.35$ s. Pick a second point $t_0 + \Delta t$; the secant slope is the finite-difference estimate of the derivative. Drag $\Delta t$ down (log scale) and watch the secant rotate onto the tangent.

In [ ]:
t0 = 0.35
h0 = h_fn(t0)
true_slope = v_fn(t0)

dt_sl = FloatLogSlider(value=0.6, base=10, min=-3, max=0,
                       description="Δt", step=0.05, readout=True,
                       readout_format=".4f")
info = Output()

t_curve = np.linspace(0, 1.47, 300)
fig, ax = plt.subplots()
ax.plot(t_curve, h_fn(t_curve), color="#2E86C1", lw=2.2, label="h(t)")
ax.scatter([t0], [h0], color="#2C3E50", s=45, zorder=6)

# tangent (for reference)
t_tan = np.linspace(t0 - 0.35, t0 + 0.35, 30)
h_tan = h0 + true_slope * (t_tan - t0)
ax.plot(t_tan, h_tan, color="#2C3E50", lw=1.6, linestyle="--",
        label=f"true tangent  (slope = {true_slope:+.3f})")

secant_line, = ax.plot([], [], color="#E67E22", lw=2.0, label="secant")
secant_pt = ax.scatter([], [], color="#E67E22", s=35, zorder=5)
ax.set_xlabel("t"); ax.set_ylabel("h(t)"); ax.legend(loc="lower center")
ax.set_xlim(-0.05, 1.55); ax.set_ylim(-0.2, 3.2)

def redraw_secant(*_):
    dt = dt_sl.value
    t1 = t0 + dt
    h1 = h_fn(t1)
    slope = (h1 - h0) / dt
    t_line = np.linspace(t0 - 0.15, t1 + 0.05, 50)
    secant_line.set_data(t_line, h0 + slope * (t_line - t0))
    secant_pt.set_offsets([[t1, h1]])
    with info:
        info.clear_output(wait=True)
        print(f"Δt        = {dt:.6f}")
        print(f"secant    = {slope:+.6f}")
        print(f"tangent   = {true_slope:+.6f}")
        print(f"error     = {abs(slope - true_slope):.2e}")
    fig.canvas.draw_idle()

dt_sl.observe(redraw_secant, names="value")
redraw_secant()
display(VBox([dt_sl, info]))

## 3 · The sweet spot — don't make Δt *too* small

The naive reasoning says "smaller Δt = better". That's true — until the floating-point subtraction $h(t_0 + \Delta t) - h(t_0)$ loses most of its significant digits to cancellation. Plot the error vs Δt and you get a V.

In [ ]:
dts = np.logspace(0, -16, 120)
errs = np.abs((h_fn(t0 + dts) - h_fn(t0)) / dts - true_slope)

fig2, ax2 = plt.subplots()
ax2.loglog(dts, errs, color="#E74C3C", lw=2.0)
ax2.axvline(1e-8, color="#7F8C8D", linestyle=":", label=r"$\sqrt{\varepsilon_{\mathrm{mach}}} \approx 10^{-8}$")
ax2.set_xlabel("Δt"); ax2.set_ylabel("error |numeric − true|")
ax2.set_title("Truncation error drops, then round-off takes over")
ax2.invert_xaxis()   # read left→right as Δt shrinks
ax2.legend()
plt.show()

idx_best = int(np.argmin(errs))
print(f"minimum error {errs[idx_best]:.2e} achieved near Δt = {dts[idx_best]:.2e}")

## 4 · Riemann sum — rectangles approach the integral

Integrate $v(t) = v_{0y} - g\,t$ from 0 to the apex $t_\star = v_{0y}/g$. The analytic answer is $v_{0y}^2/(2g) \approx 2.64$ m. Use left-endpoint rectangles so the convergence is visible (midpoint would be exact on a linear $v$).

In [ ]:
n_sl = IntSlider(value=4, min=2, max=64, step=1, description="n rects",
                 continuous_update=True)
sum_out = Output()

t_plot = np.linspace(0, t_apex, 300)
v_plot = v_fn(t_plot)
exact_integral = v0y * t_apex - 0.5 * g * t_apex ** 2

fig3, ax3 = plt.subplots()
ax3.plot(t_plot, v_plot, color="#2E86C1", lw=2.3, label="v(t)")
ax3.fill_between(t_plot, 0, v_plot, alpha=0.12, color="#2E86C1")
ax3.axhline(0, color="#7F8C8D", lw=0.6)
bars = ax3.bar([], [], width=[], align="edge", alpha=0.35, color="#E67E22",
               edgecolor="#E67E22", linewidth=0.7)
ax3.set_xlabel("t"); ax3.set_ylabel("v(t)"); ax3.set_title("left-endpoint Riemann sum")
ax3.legend(loc="upper right")

def redraw_riemann(*_):
    n = n_sl.value
    edges = np.linspace(0, t_apex, n + 1)
    widths = np.diff(edges)
    left_pts = edges[:-1]
    heights = v_fn(left_pts)
    for patch in ax3.patches:
        patch.remove()
    for xi, w, hi in zip(left_pts, widths, heights):
        ax3.bar(xi, hi, width=w, align="edge", alpha=0.35, color="#E67E22",
                edgecolor="#E67E22", linewidth=0.7)
    sum_val = float(np.sum(heights * widths))
    with sum_out:
        sum_out.clear_output(wait=True)
        print(f"n rectangles: {n:>3}")
        print(f"left-sum    : {sum_val:.6f}")
        print(f"exact       : {exact_integral:.6f}")
        print(f"error       : {abs(sum_val - exact_integral):.6f}")
    fig3.canvas.draw_idle()

n_sl.observe(redraw_riemann, names="value")
redraw_riemann()
display(VBox([n_sl, sum_out]))

## 5 · Fundamental Theorem of Calculus — two operations, one coin

FTC claims: **if you integrate the derivative you get the original function back** (up to a constant). Verify numerically at several times $T$.

In [ ]:
from scipy.integrate import quad

for T in [0.10, 0.30, 0.50, 0.734, 1.00, 1.40]:
    integral, _ = quad(v_fn, 0, T)    # integrate v(t) from 0 to T
    print(f"T = {T:.3f}   ∫₀ᵀ v(t)dt = {integral:+.6f}   h(T) − h(0) = {h_fn(T) - h_fn(0):+.6f}")

## 6 · Archimedes' polygon-exhaustion of a circle

The integral idea is older than Newton by 1900 years. Archimedes approximated the area of a unit circle using inscribed $n$-gons. As $n$ grows, the polygon area climbs toward $\pi$.

In [ ]:
n_poly = IntSlider(value=4, min=3, max=256, step=1, description="n sides",
                   continuous_update=True)
pi_out = Output()

theta = np.linspace(0, 2 * np.pi, 400)
fig4, ax4 = plt.subplots(figsize=(5.5, 5.5))
ax4.plot(np.cos(theta), np.sin(theta), color="#2C3E50", lw=2.0)
(poly_line,) = ax4.plot([], [], color="#8E44AD", lw=1.6)
ax4.set_aspect("equal"); ax4.set_xlim(-1.3, 1.3); ax4.set_ylim(-1.3, 1.3)

def redraw_poly(*_):
    n = n_poly.value
    ang = np.linspace(0, 2 * np.pi, n + 1)
    xs, ys = np.cos(ang), np.sin(ang)
    poly_line.set_data(xs, ys)
    area = 0.5 * n * np.sin(2 * np.pi / n)
    with pi_out:
        pi_out.clear_output(wait=True)
        print(f"n sides   : {n:>3}")
        print(f"polygon   : {area:.10f}")
        print(f"π (true)  : {np.pi:.10f}")
        print(f"error     : {np.pi - area:.2e}")
    fig4.canvas.draw_idle()

n_poly.observe(redraw_poly, names="value")
redraw_poly()
display(VBox([n_poly, pi_out]))

## 7 · Where this reappears

- **Pre-Req Ch.4.** The derivative tells you the direction of steepest descent. Gradient descent is calculus + the humble idea "take a small step".
- **Pre-Req Ch.6.** Chain rule → reverse-mode autodiff → `loss.backward()`.
- **ML Ch.1.** Least-squares solution = where the derivative of the loss is zero.
- **ML Ch.4/5.** Every parameter update in a neural network is $\partial L / \partial w$.
- **Probability & statistics.** Expectations, KL divergence, and variational inference are all integrals.

**One last exercise.** Modify the Riemann-sum widget (cell 4) to use **midpoint** rectangles instead of left-endpoint. How quickly does the error shrink with $n$? (Expected: for linear $v(t)$, midpoint is *exact* — that's why we deliberately used left-endpoint for visible convergence.)